In [ ]:
!pip install ultralytics gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


with 85 class


In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
from ultralytics import YOLO

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load YOLO model
# -----------------------------
yolo_model = YOLO("/content/drive/MyDrive/my_model/my_model.pt")

# -----------------------------
# Load Classification Model
# -----------------------------
num_classes = 85

classifier = models.efficientnet_b0(weights=None)
classifier.classifier[1] = nn.Linear(
    classifier.classifier[1].in_features, num_classes
)

classifier.load_state_dict(
    torch.load("/content/drive/MyDrive/my_model/traffic_sign_classifier_85.pth",
               map_location=device)
)

classifier = classifier.to(device)
classifier.eval()

print("Models Loaded Successfully")

# -----------------------------
# Image Transform
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

class_names = [
'ALL_MOTOR_VEHICLE_PROHIBITED',
'AXLE_LOAD_LIMIT',
'BARRIER_AHEAD',
'BULLOCK_AND_HANDCART_PROHIBITED',
'BULLOCK_PROHIBITED',
'CATTLE',
'COMPULSARY_AHEAD',
'COMPULSARY_AHEAD_OR_TURN_LEFT',
'COMPULSARY_AHEAD_OR_TURN_RIGHT',
'COMPULSARY_CYCLE_TRACK',
'COMPULSARY_KEEP_LEFT',
'U_TURN_PROHIBITED',
'COMPULSARY_MINIMUM_SPEED',
'COMPULSARY_SOUND_HORN',
'COMPULSARY_TURN_LEFT',
'COMPULSARY_TURN_LEFT_AHEAD',
'COMPULSARY_TURN_RIGHT',
'COMPULSARY_TURN_RIGHT_AHEAD',
'CROSS_ROAD',
'CYCLE_CROSSING',
'CYCLE_PROHIBITED',
'DANGEROUS_DIP',
'DIRECTION',
'FALLING_ROCKS',
'FERRY',
'GAP_IN_MEDIAN',
'GIVE_WAY',
'GUARDED_LEVEL_CROSSING',
'HANDCART_PROHIBITED',
'HEIGHT_LIMIT',
'HORN_PROHIBITED',
'HUMP_OR_ROUGH_ROAD',
'LEFT_HAIR_PIN_BEND',
'LEFT_HAND_CURVE',
'LEFT_REVERSE_BEND',
'LEFT_TURN_PROHIBITED',
'LENGTH_LIMIT',
'LOAD_LIMIT',
'LOOSE_GRAVEL',
'MEN_AT_WORK',
'NARROW_BRIDGE',
'NARROW_ROAD_AHEAD',
'NO_ENTRY',
'NO_PARKING',
'NO_STOPPING_OR_STANDING',
'OVERTAKING_PROHIBITED',
'PASS_EITHER_SIDE',
'PEDESTRIAN_CROSSING',
'PEDESTRIAN_PROHIBITED',
'PRIORITY_FOR_ONCOMING_VEHICLES',
'QUAY_SIDE_OR_RIVER_BANK',
'RESTRICTION_ENDS',
'RIGHT_HAIR_PIN_BEND',
'RIGHT_HAND_CURVE',
'RIGHT_REVERSE_BEND',
'RIGHT_TURN_PROHIBITED',
'ROAD_WIDENS_AHEAD',
'ROUNDABOUT',
'SCHOOL_AHEAD',
'SIDE_ROAD_LEFT',
'SIDE_ROAD_RIGHT',
'SLIPPERY_ROAD',
'SPEED_LIMIT_15',
'SPEED_LIMIT_20',
'SPEED_LIMIT_30',
'SPEED_LIMIT_40',
'SPEED_LIMIT_5',
'SPEED_LIMIT_50',
'SPEED_LIMIT_60',
'SPEED_LIMIT_70',
'SPEED_LIMIT_80',
'STAGGERED_INTERSECTION',
'STEEP_ASCENT',
'STEEP_DESCENT',
'STOP',
'STRAIGHT_PROHIBITED',
'TONGA_PROHIBITED',
'TRAFFIC_SIGNAL',
'TRUCK_PROHIBITED',
'TURN_RIGHT',
'T_INTERSECTION',
'UNGUARDED_LEVEL_CROSSING',
'U_TURN_PROHIBITED',
'WIDTH_LIMIT',
'Y_INTERSECTION'
]

# -----------------------------
# Prediction Function
# -----------------------------
def detect_and_classify(image):

    image = np.array(image)

    # YOLO detection
    results = yolo_model(image)

    boxes = results[0].boxes

    # Annotated image from YOLO (keep original colors)
    annotated_image = results[0].plot()

    predictions = []

    for box in boxes:

        if float(box.conf[0]) < 0.5:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        roi = image[y1:y2, x1:x2]

        if roi.size == 0:
            continue

        roi_pil = Image.fromarray(roi)

        input_tensor = transform(roi_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = classifier(input_tensor)
            _, predicted = torch.max(outputs,1)

        class_name = class_names[predicted.item()]

        predictions.append(class_name)

    if len(predictions) == 0:
        result_text = "No traffic signs detected"
    else:
        result_text = "\n".join(predictions)

    return annotated_image, result_text


# -----------------------------
# Gradio UI
# -----------------------------
interface = gr.Interface(
    fn=detect_and_classify,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="YOLO Detection Output"),
        gr.Textbox(label="Classification Results")
    ],
    title="Traffic Sign Detection and Classification",
    description="Upload an image → YOLO detects traffic signs → EfficientNet classifies them"
)

interface.launch()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Models Loaded Successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6983eb2220d47042fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


with 43 classes

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
from ultralytics import YOLO

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load YOLO model
# -----------------------------
yolo_model = YOLO("/content/drive/MyDrive/my_model/my_model.pt")

# -----------------------------
# Load Classification Model
# -----------------------------
num_classes = 43

classifier = models.efficientnet_b0(weights=None)
classifier.classifier[1] = nn.Linear(
    classifier.classifier[1].in_features, num_classes
)

classifier.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/my_model/traffic_sign_model.pth",
        map_location=device
    )
)

classifier = classifier.to(device)
classifier.eval()

print("Models Loaded Successfully")

# -----------------------------
# Image Transform (FIXED)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# -----------------------------
# Class Labels
# -----------------------------
classes = {
    0: 'Speed limit (20km/h)',
    1: 'Speed limit (30km/h)',
    2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)',
    5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',
    7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',
    9: 'No passing',
    10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Vehicles over 3.5 metric tons prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows on the right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed and passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End of no passing by vehicles over 3.5 metric tons'
}

# -----------------------------
# Prediction Function
# -----------------------------
def detect_and_classify(image):
    try:
        image = np.array(image)

        # YOLO detection
        results = yolo_model(image)
        annotated_image = results[0].plot()

        boxes = results[0].boxes

        # If no detections
        if boxes is None or len(boxes) == 0:
            return annotated_image, "No traffic signs detected"

        predictions = []

        for box in boxes:
            if float(box.conf[0]) < 0.5:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Boundary safety
            h, w, _ = image.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            roi = image[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            roi_pil = Image.fromarray(roi)

            input_tensor = transform(roi_pil).unsqueeze(0).to(device)

            # Classification
            with torch.no_grad():
                outputs = classifier(input_tensor)
                _, predicted = torch.max(outputs, 1)

            class_name = classes[predicted.item()]  # ✅ FIXED
            predictions.append(class_name)

        if len(predictions) == 0:
            result_text = "No valid traffic signs detected"
        else:
            result_text = "\n".join(predictions)

        return annotated_image, result_text

    except Exception as e:
        return None, f"ERROR: {str(e)}"

# -----------------------------
# Gradio UI
# -----------------------------
interface = gr.Interface(
    fn=detect_and_classify,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="YOLO Detection Output"),
        gr.Textbox(label="Classification Results")
    ],
    title="Traffic Sign Detection and Classification",
    description="Upload an image → YOLO detects traffic signs → EfficientNet classifies them"
)

interface.launch()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Models Loaded Successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0448b52918ac2990c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
from ultralytics import YOLO

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load YOLO model
# -----------------------------
yolo_model = YOLO("/content/drive/MyDrive/my_model/my_model.pt")

# -----------------------------
# Load Classification Model
# -----------------------------
num_classes = 85

classifier = models.efficientnet_b0(weights=None)
classifier.classifier[1] = nn.Linear(
    classifier.classifier[1].in_features, num_classes
)

classifier.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/my_model/traffic_sign_classifier_85.pth",
        map_location=device
    )
)

classifier = classifier.to(device)
classifier.eval()

print("Models Loaded Successfully")

# -----------------------------
# Image Transform (FIXED)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# -----------------------------
# Class Labels
# -----------------------------


class_names = [
'ALL_MOTOR_VEHICLE_PROHIBITED',
'AXLE_LOAD_LIMIT',
'BARRIER_AHEAD',
'BULLOCK_AND_HANDCART_PROHIBITED',
'BULLOCK_PROHIBITED',
'CATTLE',
'COMPULSARY_AHEAD',
'COMPULSARY_AHEAD_OR_TURN_LEFT',
'COMPULSARY_AHEAD_OR_TURN_RIGHT',
'COMPULSARY_CYCLE_TRACK',
'COMPULSARY_KEEP_LEFT',
'COMPULSARY_KEEP_RIGHT',
'COMPULSARY_MINIMUM_SPEED',
'COMPULSARY_SOUND_HORN',
'COMPULSARY_TURN_LEFT',
'COMPULSARY_TURN_LEFT_AHEAD',
'COMPULSARY_TURN_RIGHT',
'COMPULSARY_TURN_RIGHT_AHEAD',
'CROSS_ROAD',
'CYCLE_CROSSING',
'CYCLE_PROHIBITED',
'DANGEROUS_DIP',
'DIRECTION',
'FALLING_ROCKS',
'FERRY',
'GAP_IN_MEDIAN',
'GIVE_WAY',
'GUARDED_LEVEL_CROSSING',
'HANDCART_PROHIBITED',
'HEIGHT_LIMIT',
'HORN_PROHIBITED',
'HUMP_OR_ROUGH_ROAD',
'LEFT_HAIR_PIN_BEND',
'LEFT_HAND_CURVE',
'LEFT_REVERSE_BEND',
'LEFT_TURN_PROHIBITED',
'LENGTH_LIMIT',
'LOAD_LIMIT',
'LOOSE_GRAVEL',
'MEN_AT_WORK',
'NARROW_BRIDGE',
'NARROW_ROAD_AHEAD',
'NO_ENTRY',
'NO_PARKING',
'NO_STOPPING_OR_STANDING',
'OVERTAKING_PROHIBITED',
'PASS_EITHER_SIDE',
'PEDESTRIAN_CROSSING',
'PEDESTRIAN_PROHIBITED',
'PRIORITY_FOR_ONCOMING_VEHICLES',
'QUAY_SIDE_OR_RIVER_BANK',
'RESTRICTION_ENDS',
'RIGHT_HAIR_PIN_BEND',
'RIGHT_HAND_CURVE',
'RIGHT_REVERSE_BEND',
'RIGHT_TURN_PROHIBITED',
'ROAD_WIDENS_AHEAD',
'ROUNDABOUT',
'SCHOOL_AHEAD',
'SIDE_ROAD_LEFT',
'SIDE_ROAD_RIGHT',
'SLIPPERY_ROAD',
'SPEED_LIMIT_15',
'SPEED_LIMIT_20',
'SPEED_LIMIT_30',
'SPEED_LIMIT_40',
'SPEED_LIMIT_5',
'SPEED_LIMIT_50',
'SPEED_LIMIT_60',
'SPEED_LIMIT_70',
'SPEED_LIMIT_80',
'STAGGERED_INTERSECTION',
'STEEP_ASCENT',
'STEEP_DESCENT',
'STOP',
'STRAIGHT_PROHIBITED',
'TONGA_PROHIBITED',
'TRAFFIC_SIGNAL',
'TRUCK_PROHIBITED',
'TURN_RIGHT',
'T_INTERSECTION',
'UNGUARDED_LEVEL_CROSSING',
'U_TURN_PROHIBITED',
'WIDTH_LIMIT',
'Y_INTERSECTION'
]

# -----------------------------
# Prediction Function
# -----------------------------
def detect_and_classify(image):
    try:
        image = np.array(image)

        # YOLO detection
        results = yolo_model(image)
        annotated_image = results[0].plot()

        boxes = results[0].boxes

        # If no detections
        if boxes is None or len(boxes) == 0:
            return annotated_image, "No traffic signs detected"

        predictions = []

        for box in boxes:
            if float(box.conf[0]) < 0.5:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Boundary safety
            h, w, _ = image.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)

            roi = image[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            roi_pil = Image.fromarray(roi)

            input_tensor = transform(roi_pil).unsqueeze(0).to(device)

            # Classification
            with torch.no_grad():
                outputs = classifier(input_tensor)
                _, predicted = torch.max(outputs, 1)

            class_names = classes[predicted.item()]  # ✅ FIXED
            predictions.append(class_names)

        if len(predictions) == 0:
            result_text = "No valid traffic signs detected"
        else:
            result_text = "\n".join(predictions)

        return annotated_image, result_text

    except Exception as e:
        return None, f"ERROR: {str(e)}"

# -----------------------------
# Gradio UI
# -----------------------------
interface = gr.Interface(
    fn=detect_and_classify,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="YOLO Detection Output"),
        gr.Textbox(label="Classification Results")
    ],
    title="Traffic Sign Detection and Classification",
    description="Upload an image → YOLO detects traffic signs → EfficientNet classifies them"
)

interface.launch()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Models Loaded Successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d938b943128d9078b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
